# Gridlock Hackathon 2.0 — Traffic Demand Prediction

**Team**: nishant.rajpathak06  
**Best LB Score**: 86.01 (`submission_calibrated.csv`)

### Approach
LightGBM with 5-fold `TimeSeriesSplit` CV, extensive leave-one-out target encodings, and a cross-day calibration factor.

**Key findings:**
- LOO target encodings (geohash×hour, geohash×tod, etc.) gave the single largest boost — +22 pt over lag-only baseline
- `RoadType_le` is the dominant feature (~60% gain importance)
- `lag_1` (15-min lag) inflates CV by ~3 pt — ablation removes it for an honest LB proxy
- Per-geohash day49/day48 calibration (alpha=0.2) adds +1 pt on LB

### Pipeline
```
Raw CSV → build_base_features (datetime, spatial, lags)
        → add_all_encodings (LOO TEs per fold)
        → LightGBM 5-fold CV  →  average fold predictions
        → calibrate with day49/day48 scale
        → submission_calibrated.csv  (LB 86.01)
```

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

ROOT       = os.getcwd()          # must be the project root (c:/Flipkart)
DATA_DIR   = os.path.join(ROOT, 'dataset')
MODELS_DIR = os.path.join(ROOT, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)
sys.path.insert(0, ROOT)

print(f'ROOT     : {ROOT}')
print(f'LightGBM : {lgb.__version__}')

## 1. Load data

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

print(f'train : {train_df.shape}   test : {test_df.shape}')
print(f'days  : {sorted(train_df["day"].unique())}  (48=full, 49=00:00-02:00 in train / 02:15-13:45 in test)')
print(f'target demand — min={train_df.demand.min():.4f}  max={train_df.demand.max():.4f}  mean={train_df.demand.mean():.4f}')
print(f'\nMissing values in train:')
print(train_df.isnull().sum()[train_df.isnull().sum() > 0])

## 2. Feature engineering

All logic lives in `scripts/features.py`. Key groups:

| Group | Count | Notes |
|-------|-------|-------|
| Datetime | 13 | hour, minute, time_of_day (0-95), sin/cos cyclicals, day_mod7, is_weekend, tod_bucket |
| Spatial | 5 | lat, lon decoded from geohash, prefix4_le, prefix5_le, geohash_le |
| Road/traffic | 9 | RoadType_le, NumberofLanes, LargeVehicles_le, Landmarks_le, Weather_le, Temperature, roadtype_hour_le, weather_hour_le, roadtype_was_missing |
| Lag features | 5 | lag_1/2/4 (short-term), lag_96 (24h same time yesterday), lag_192 (48h) |
| Rolling | 4 | roll_mean/std × windows (4, 96) |
| LOO TEs | 16 | 8 group specs × (mean + median): geohash, gh×hour, gh×tod, gh×weekend, p5×hour, p5×tod_bucket, NLanes×hour, LargeVeh×hour |
| Smoothed TE | 1 | RoadType (Bayesian smoothed) |
| Stat features | 2 | gh_count, gh_hr_std |
| Cross-day | 2 | lag96_dev (deviation from expected), gh_day49_scale (day49/day48 ratio) |
| **Total** | **57** | |

In [ ]:
from scripts.features import (
    build_base_features, add_all_encodings, compute_day49_scale,
    ALL_COLS_V2, SHORT_LAG_COLS, CAT_FEATURE_NAMES,
)
from scripts.cv import get_folds, print_fold_info, N_SPLITS

print('Building base features...')
train_feat, test_feat, _, _ = build_base_features(train_df, test_df)
y = train_feat['demand'].values.astype(np.float64)

# Per-geohash day49/day48 scale ratio (used for calibration and as a feature)
day49_scale = compute_day49_scale(train_feat, y)
print(f'day49_scale: {len(day49_scale)} geohashes  mean={day49_scale.mean():.3f}  median={day49_scale.median():.3f}')

# Feature sets
USE_COLS      = ALL_COLS_V2                                        # 57 features
ABLATION_COLS = [c for c in USE_COLS if c not in SHORT_LAG_COLS]  # 54 (no lag_1/2/4)
cat_idx       = [USE_COLS.index(c) for c in CAT_FEATURE_NAMES if c in USE_COLS]
cat_idx_abl   = [ABLATION_COLS.index(c) for c in CAT_FEATURE_NAMES if c in ABLATION_COLS]

print(f'\nFull feature set       : {len(USE_COLS)} features')
print(f'Ablation (no lag_1/2/4): {len(ABLATION_COLS)} features')

## 3. CV folds

`TimeSeriesSplit` with n=5 respects temporal order — each fold's training always precedes its validation.

In [ ]:
folds = get_folds(train_feat, N_SPLITS)
print_fold_info(train_feat, folds)

## 4. Train LightGBM

- **RUN 1** (full features): saves fold models, used for submission
- **RUN 2** (ablation — no lag_1/2/4): honest LB proxy

> **Why ablation?** `lag_1/2/4` are mostly NaN for test rows (test demand is unknown at prediction time), so they inflate training CV by ~3 pt but don't help on LB. The ablation CV is the honest estimate.

In [ ]:
# Best known params — Run 6, LB 86.01
LGB_PARAMS = {
    'objective':         'regression',
    'metric':            'rmse',
    'num_leaves':        127,
    'learning_rate':     0.05,
    'feature_fraction':  0.8,
    'bagging_fraction':  0.8,
    'bagging_freq':      5,
    'min_child_samples': 20,
    'lambda_l1':         0.1,
    'lambda_l2':         0.1,
    'verbose':           -1,
    'seed':              42,
    'n_jobs':            -1,
}
NUM_ROUNDS = 3000
ES_ROUNDS  = 100


def train_cv(use_cols, cat_idx_list, label, save_models=False):
    """5-fold CV. Returns (oof_preds, test_preds_avg, oof_r2)."""
    oof     = np.full(len(train_feat), np.nan)
    t_preds = []
    fold_r2s = []

    for fold_i, (tr_idx, val_idx) in enumerate(folds):
        tr_raw  = train_feat.iloc[tr_idx].reset_index(drop=True)
        val_raw = train_feat.iloc[val_idx].reset_index(drop=True)
        y_tr    = pd.Series(y[tr_idx])
        y_val   = pd.Series(y[val_idx])

        tr_enc, val_enc, te_enc = add_all_encodings(
            tr_raw, y_tr, val_raw, test_feat, day49_scale=day49_scale)

        X_tr   = tr_enc[use_cols].values
        X_val  = val_enc[use_cols].values
        X_test = te_enc[use_cols].values

        ds_tr  = lgb.Dataset(X_tr,  label=y_tr.values,  categorical_feature=cat_idx_list, free_raw_data=False)
        ds_val = lgb.Dataset(X_val, label=y_val.values, categorical_feature=cat_idx_list,
                             free_raw_data=False, reference=ds_tr)

        model = lgb.train(
            LGB_PARAMS, ds_tr,
            num_boost_round=NUM_ROUNDS,
            valid_sets=[ds_tr, ds_val],
            valid_names=['train', 'val'],
            callbacks=[
                lgb.early_stopping(ES_ROUNDS, verbose=False),
                lgb.log_evaluation(500),
            ],
        )

        val_pred = model.predict(X_val, num_iteration=model.best_iteration)
        fold_r2  = r2_score(y_val.values, val_pred)
        fold_r2s.append(fold_r2)
        print(f'  [{label}] fold {fold_i+1}  iter={model.best_iteration:4d}  R²={fold_r2*100:.4f}%')

        oof[val_idx] = val_pred
        t_preds.append(model.predict(X_test, num_iteration=model.best_iteration))
        if save_models:
            model.save_model(os.path.join(MODELS_DIR, f'fold_{fold_i}.lgb'))

    mask   = ~np.isnan(oof)
    oof_r2 = r2_score(y[mask], oof[mask])
    test_p = np.clip(np.mean(t_preds, axis=0), 0.0, 1.0)
    print(f'\n  [{label}] OOF R² = {oof_r2*100:.4f}%  '
          f'(covers {mask.mean()*100:.1f}% of train)')
    print(f'  Fold R²s: {[f"{r*100:.2f}" for r in fold_r2s]}')
    return oof, test_p, oof_r2

In [ ]:
print('=' * 60)
print('RUN 1 — Full features (saves models for submission)')
print('=' * 60)
oof_full, test_full, r2_full = train_cv(
    USE_COLS, cat_idx, 'full', save_models=True)

np.save(os.path.join(MODELS_DIR, 'oof_preds.npy'), oof_full)
np.save(os.path.join(MODELS_DIR, 'test_preds.npy'), test_full)

In [ ]:
print('=' * 60)
print('RUN 2 — Ablation: no lag_1/2/4  (honest LB proxy)')
print('=' * 60)
oof_abl, test_abl, r2_abl = train_cv(
    ABLATION_COLS, cat_idx_abl, 'ablation', save_models=False)

np.save(os.path.join(MODELS_DIR, 'oof_ablation_preds.npy'), oof_abl)
print(f'\nShort-lag CV inflation: {(r2_full - r2_abl)*100:+.2f} pt')

## 5. Feature importances

In [ ]:
imp = np.zeros(len(USE_COLS))
for i in range(N_SPLITS):
    m = lgb.Booster(model_file=os.path.join(MODELS_DIR, f'fold_{i}.lgb'))
    imp += m.feature_importance(importance_type='gain')
imp /= N_SPLITS

imp_df = pd.DataFrame({'feature': USE_COLS, 'gain': imp})
imp_df = imp_df.sort_values('gain', ascending=False)
imp_df['pct'] = imp_df['gain'] / imp_df['gain'].sum() * 100
print('Top 20 features by gain importance:')
print(imp_df.head(20).to_string(index=False))

## 6. Generate submissions

**`submission.csv`** — raw average of 5 fold model predictions (LB 85.04)  
**`submission_calibrated.csv`** — multiplied by per-geohash day49/day48 scale at alpha=0.2 (**LB 86.01** — best)

In [ ]:
# Build test features using all training data for TEs (no fold split)
_, test_enc_full, _ = add_all_encodings(
    train_feat, pd.Series(y), test_feat, None, day49_scale=day49_scale)

models   = [lgb.Booster(model_file=os.path.join(MODELS_DIR, f'fold_{i}.lgb'))
            for i in range(N_SPLITS)]
X_final  = test_enc_full[USE_COLS].values
raw_pred = np.clip(np.mean([m.predict(X_final) for m in models], axis=0), 0.0, 1.0)

print(f'Raw predictions: mean={raw_pred.mean():.5f}  '
      f'min={raw_pred.min():.5f}  max={raw_pred.max():.5f}')

sub = pd.DataFrame({'Index': test_df['Index'].values, 'demand': raw_pred})
assert sub.shape == (41778, 2) and list(sub.columns) == ['Index', 'demand']
sub.to_csv('submission.csv', index=False)
print('Written: submission.csv')

In [ ]:
# Calibrated submission — best LB 86.01
ALPHA        = 0.2   # confirmed best alpha on LB
global_scale = float(day49_scale.mean())
scale_test   = test_feat['geohash'].map(day49_scale).fillna(global_scale).values
multiplier   = ALPHA * scale_test + (1.0 - ALPHA)
calib_pred   = np.clip(raw_pred * multiplier, 0.0, 1.0)

print(f'Calibrated:  mean={calib_pred.mean():.5f}  '
      f'multiplier mean={multiplier.mean():.4f}')

sub_c = pd.DataFrame({'Index': test_df['Index'].values, 'demand': calib_pred})
assert sub_c.shape == (41778, 2)
sub_c.to_csv('submission_calibrated.csv', index=False)
print('Written: submission_calibrated.csv  ← BEST SUBMISSION (LB 86.01)')

## 7. Verify output

In [ ]:
for fname in ['submission.csv', 'submission_calibrated.csv']:
    df = pd.read_csv(fname)
    ok = (df.shape == (41778, 2)
          and list(df.columns) == ['Index', 'demand']
          and df['demand'].between(0, 1).all()
          and df['Index'].equals(test_df['Index']))
    print(f'{fname}:  shape={df.shape}  '
          f'demand=[{df.demand.min():.4f}, {df.demand.max():.4f}]  '
          f'mean={df.demand.mean():.5f}  valid={ok}')

## 8. Experiment log

| # | Description | Ablation CV | LB Score |
|---|-------------|-------------|----------|
| 1 | LightGBM baseline (minimal features) | 64.82% | — |
| 2 | + lag/rolling features | 67.19% | — |
| 3 | + LOO target encodings (8 group specs) | **85.68%** | — |
| 4 | + more TEs + spatial (prefix4, tod_bucket) | 86.73% | — |
| 5 | + RoadType×time TEs (regression) | 78.30% | — |
| 6 | Restored Run4 + RoadType NaN imputation | 87.68% | 85.04 |
| 7 | Run6 + cross-day calibration alpha=0.2 | 87.68% | **86.01** ← best |

**What did NOT help (tried and discarded):**
- XGBoost / CatBoost ensembles: error correlation with LGB = 0.937, blending always hurt OOF
- Seed averaging: single seed 42 was already optimal
- Autoregressive lag filling: correlation with standard pred = 0.991, no LB gain
- Lag_96 post-processing (scale from 0:00–2:00 night doesn't transfer to 2:15–13:45 rush hours, LB 63–74)
- Global RoadType×time TEs: training/validation inconsistency (LOO-corrected training vs raw global val) cost ~4 pt
- num_leaves=255: overfits small early folds (fold 1 has only 12k rows), lost ~4 pt on ablation CV

**Root cause of the CV↔LB gap (~1.7 pt):**  
All 5 CV folds train exclusively on day48. `lag_96` (r=0.792) is always NaN in training — LGB cannot learn its coefficient. Test is day49 where lag_96 is valid for 89% of rows but the model treats it as missing.